# Step 4 — Robustness of the "generic fold" result

**Step 3 result.** Balanced MNIST pairs (0,1) and (3,8) showed *folds* (birth jump ≈ 1.0), not pitchforks, along the class-mean direction. Some pairs ((4,9)) were never bimodal along that direction.

**This notebook asks three questions.**
1. **Direction choice.** Is the fold an artifact of the mean-difference direction? Repeat with the Fisher/LDA direction `u ∝ (S_w + λI)^{-1}(μ_a − μ_b)`, which accounts for within-class covariance.
2. **Population statistics.** Over **all 45 balanced digit pairs**, what is the distribution of birth jumps? How many are folds, pitchforks, or never bimodal?
3. **Detector mismatch on real data.** Compute `Var[H(c|m)]` exactly from the per-class 1-D marginals (no network) and compare its peak τ with the bifurcation τ, as in Step 2.

CPU, ~5 minutes. Internet on for the MNIST download.

In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
rng = np.random.default_rng(0)
plt.rcParams['figure.dpi'] = 110

ds = datasets.MNIST('/kaggle/working/mnist', train=True, download=True)
X = (ds.data.numpy().astype(np.float32) / 127.5 - 1.0).reshape(-1, 784)
Y = ds.targets.numpy()
print(X.shape)

In [ ]:
GRID = np.linspace(-4, 4, 4001); DG = GRID[1] - GRID[0]
N_PER = 10000

def direction(Xa, Xb, kind, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a - mu_b
    if kind == 'mean':
        u = delta
    elif kind == 'lda':
        Sw = np.cov(Xa.T) + np.cov(Xb.T)
        u = np.linalg.solve(Sw + lam*np.eye(len(delta)), delta)
    else:
        raise ValueError
    return u / np.linalg.norm(u), 0.5*(mu_a + mu_b)

def project(Xa, Xb, w, kind):
    """Return per-class projected samples (ma, mb), scaled so class means are at +/-1, and the scale factor."""
    u, mid = direction(Xa, Xb, kind)
    pa = (Xa - mid) @ u; pb = (Xb - mid) @ u
    half = 0.5*(pa.mean() - pb.mean())          # >0
    pa, pb = pa/half, pb/half
    na = int(w*2*N_PER); nb_ = 2*N_PER - na
    A = pa[rng.choice(len(pa), na, replace=len(pa) < na)]
    B = pb[rng.choice(len(pb), nb_, replace=len(pb) < nb_)]
    return A, B, half   # sigma -> tau: tau = sigma^2 / half^2

def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID - DG/2, GRID[-1] + DG/2), density=True)
    k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(np.convolve(h, k, mode='same'), 1e-300)

def stationary_points(p, merge=0.02):
    d = np.gradient(np.log(p), GRID)
    keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    pts = [(float(GRID[i] - d[i]*DG/(d[i+1]-d[i])) if d[i+1] != d[i] else float(GRID[i]),
            'max' if d[i] > 0 else 'min') for i in idx]
    changed = True
    while changed and len(pts) > 1:
        changed = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]:
                del pts[a:a+2]; changed = True; break
    return pts

def birth_jump(diag, taus):
    prev_max = None
    for tau in sorted(taus, reverse=True):
        mx = sorted(p for p, k in diag[tau] if k == 'max'); mn = [p for p, k in diag[tau] if k == 'min']
        if len(mx) >= 2 and prev_max is not None:
            c = mn[0] if mn else 0.5*(mx[0]+mx[-1])
            return tau, abs(c - prev_max)
        if len(mx) == 1: prev_max = mx[0]
    return None

def varH_curve(A, B, w, taus, n=100_000):
    """Var over samples of H(c|m), computed from exact per-class 1-D marginals."""
    out = []
    for tau in taus:
        pa, pb = smooth_hist(A, tau), smooth_hist(B, tau)
        m = np.concatenate([A[rng.choice(len(A), int(w*n))], B[rng.choice(len(B), n-int(w*n))]]) \
            + np.sqrt(tau)*rng.standard_normal(n)
        ia = np.clip(((m - GRID[0])/DG).round().astype(int), 0, len(GRID)-1)
        pc = w*pa[ia] / (w*pa[ia] + (1-w)*pb[ia])
        pc = np.clip(pc, 1e-12, 1-1e-12)
        H = -(pc*np.log(pc) + (1-pc)*np.log(1-pc))
        out.append(H.var())
    return np.array(out)

TAUS = np.geomspace(0.05, 3.0, 60)

## 1 + 2. All 45 balanced pairs, two directions

In [ ]:
rows = []
for a, b in combinations(range(10), 2):
    Xa, Xb = X[Y == a], X[Y == b]
    rec = {'pair': (a, b)}
    for kind in ['mean', 'lda']:
        A, B, half = project(Xa, Xb, 0.5, kind)
        m0 = np.concatenate([A, B])
        diag = {t: stationary_points(smooth_hist(m0, t)) for t in TAUS}
        bj = birth_jump(diag, TAUS)
        rec[kind] = bj
        rec[kind+'_sep'] = float(abs(A.mean()-B.mean()) / np.sqrt(0.5*(A.var()+B.var())))  # class separability along u
    rows.append(rec)
    f = lambda bj: f"tau={bj[0]:.3f} jump={bj[1]:.2f}" if bj else "never bimodal   "
    print(f"{(a,b)}  mean-dir: {f(rec['mean'])}  (sep {rec['mean_sep']:.2f})   lda-dir: {f(rec['lda'])}  (sep {rec['lda_sep']:.2f})")

for kind in ['mean', 'lda']:
    js = [r[kind][1] for r in rows if r[kind]]
    nb_ = sum(r[kind] is None for r in rows)
    print(f"\n[{kind}] bimodal pairs: {len(js)}/45, never bimodal: {nb_};  birth jump median={np.median(js):.2f}, "
          f"min={min(js):.2f}, max={max(js):.2f};  pairs with jump<0.1 (pitchfork-like): {sum(j < 0.1 for j in js)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, kind in zip(axes, ['mean', 'lda']):
    js = [r[kind][1] for r in rows if r[kind]]
    ax.hist(js, bins=np.linspace(0, 2.0, 21), alpha=0.8)
    ax.axvline(0.0, color='k', ls='--', label='pitchfork (control, w=0.5)')
    ax.axvline(1.6, color='C3', ls='--', label='fold (control, w=0.8)')
    ax.set_title(f'{kind} direction: birth jumps over balanced pairs (n={len(js)})'); ax.set_xlabel('birth jump'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# does the jump grow with the shape asymmetry between the two class humps? (ratio of class stds along u)
fig, ax = plt.subplots(figsize=(5, 3.6))
for kind, col in [('mean', 'C0'), ('lda', 'C1')]:
    xs, ys = [], []
    for r in rows:
        if r[kind]:
            a, b = r['pair']
            A, B, _ = project(X[Y == a], X[Y == b], 0.5, kind)
            xs.append(max(A.std(), B.std())/min(A.std(), B.std())); ys.append(r[kind][1])
    ax.scatter(xs, ys, s=18, color=col, label=kind)
ax.set_xlabel('width ratio of the two class humps'); ax.set_ylabel('birth jump'); ax.legend()
plt.tight_layout(); plt.show()

## 3. Var[H(c|m)] peak vs bifurcation on real data (LDA direction, balanced)

In [ ]:
print(f"{'pair':>7} {'tau_bif':>8} {'tau_varH_peak':>14} {'ratio':>6}")
pts = []
for r in rows:
    if not r['lda']: continue
    a, b = r['pair']
    A, B, _ = project(X[Y == a], X[Y == b], 0.5, 'lda')
    v = varH_curve(A, B, 0.5, TAUS, n=40_000)
    tp = TAUS[np.argmax(v)]; tb = r['lda'][0]
    pts.append((tb, tp)); print(f"{(a,b)!s:>7} {tb:8.3f} {tp:14.3f} {tp/tb:6.2f}")

fig, ax = plt.subplots(figsize=(4.5, 4))
tb, tp = np.array(pts).T
ax.scatter(tb, tp, s=18); lim = [min(tb.min(), tp.min())*0.8, max(tb.max(), tp.max())*1.2]
ax.plot(lim, lim, 'k--', lw=0.8); ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('bifurcation tau (fold)'); ax.set_ylabel('Var[H] peak tau'); ax.set_title('MNIST pairs: detector vs bifurcation')
plt.tight_layout(); plt.show()

## What to report back

1. The 45-pair table and the two histograms. Key numbers: how many balanced pairs are folds (jump clearly > 0.1) under each direction, and how many are never bimodal. If the LDA direction rescues pairs like (4,9), say so.
2. The width-ratio scatter: does the jump increase with hump asymmetry?
3. The Var[H] scatter: do the points sit off the diagonal, and in which direction?